In [2]:
# Step 1 – Install Academic Research Tools
%pip install -q \
  llama-index \
  llama-index-llms-replicate \
  llama-index-embeddings-huggingface \
  llama-index-readers-file \
  llama-index-packs-fusion-retriever \
  llama-index-vector-stores-chroma \
  chromadb \
  sentence-transformers \
  huggingface_hub[hf_xet] \
  hf_xet \
  certifi \
  python-certifi-win32 \
  truststore \
  nest-asyncio \
  requests \
  replicate \
  pytesseract \
  pdf2image \
  Pillow \
  PyMuPDF

import nest_asyncio
nest_asyncio.apply()
print("✅ Installation complete (including OCR + persistent memory deps).")

# NOTE: System dependencies required for OCR:
# - Tesseract OCR executable (installable from https://github.com/tesseract-ocr/tesseract)
# - Poppler utils (for pdf2image) available via package managers or https://poppler.freedesktop.org/
# If those executables are not installed, the OCR fallback will print instructions.

Note: you may need to restart the kernel to use updated packages.
✅ Installation complete (including OCR + persistent memory deps).



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Console / Logger helper (widget-free for VS Code compatibility)
from datetime import datetime
import logging
import glob
import os
import shutil

def console_log(msg, level='INFO'):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{ts}] {level}: {msg}")

def resolve_tesseract_cmd() -> str | None:
    """Find and configure a usable Tesseract executable path."""
    candidates = []

    env_cmd = os.getenv("TESSERACT_CMD", "").strip()
    if env_cmd and os.path.exists(env_cmd):
        candidates.append(env_cmd)

    path_cmd = shutil.which("tesseract")
    if path_cmd:
        candidates.append(path_cmd)

    if os.name == "nt":
        default_paths = [
            r"C:\Program Files\Tesseract-OCR\tesseract.exe",
            r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
        ]
        for p in default_paths:
            if os.path.exists(p):
                candidates.append(p)

        local_app_data = os.getenv("LOCALAPPDATA", "")
        if local_app_data:
            winget_pattern = os.path.join(
                local_app_data,
                "Microsoft",
                "WinGet",
                "Packages",
                "*",
                "**",
                "tesseract.exe",
            )
            winget_matches = glob.glob(winget_pattern, recursive=True)
            candidates.extend(sorted(winget_matches))

    for cmd in candidates:
        if os.path.exists(cmd):
            os.environ["TESSERACT_CMD"] = cmd
            try:
                import pytesseract
                pytesseract.pytesseract.tesseract_cmd = cmd
            except Exception:
                pass
            return cmd

    return None

class ConsoleHandler(logging.Handler):
    def emit(self, record):
        console_log(self.format(record), record.levelname)

root_logger = logging.getLogger()
if not any(isinstance(h, ConsoleHandler) for h in root_logger.handlers):
    handler = ConsoleHandler()
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

root_logger.setLevel(logging.INFO)
console_log("Console initialized - logs will appear in cell output.", "OK")

[2026-04-14 07:49:13] OK: Console initialized - logs will appear in cell output.


In [4]:
# Diagnostics: verify Tesseract and Poppler (pdf2image) availability
import shutil
import os

console_log("Running diagnostics: checking system OCR dependencies...", "INFO")

# Tesseract check
tess_path = resolve_tesseract_cmd()
if tess_path:
    try:
        import pytesseract
        v = pytesseract.get_tesseract_version()
        console_log(f"Tesseract found: {tess_path} - version {v}", "OK")
    except Exception as e:
        console_log(f"Tesseract found at {tess_path} but pytesseract error: {e}", "WARN")
else:
    console_log("Tesseract executable not found in PATH or common install locations.", "ERROR")
    console_log(
        "Install Tesseract: https://github.com/tesseract-ocr/tesseract, `winget install UB-Mannheim.TesseractOCR`, or `choco install tesseract`",
        "INFO",
    )

# Poppler check (pdftoppm or pdftocairo)
poppler_bin = shutil.which("pdftoppm") or shutil.which("pdftocairo")
if poppler_bin:
    console_log(f"Poppler utility found: {poppler_bin}", "OK")
else:
    console_log("Poppler utilities (pdftoppm/pdftocairo) not found in PATH.", "ERROR")
    console_log("Install Poppler: https://poppler.freedesktop.org/ or `choco install poppler`", "INFO")

# Optional pdf2image test if a sample PDF exists
sample_pdf = os.path.join("academic_data", "source_material.pdf")
if os.path.exists(sample_pdf):
    if poppler_bin:
        try:
            from pdf2image import convert_from_path
            imgs = convert_from_path(sample_pdf, dpi=50, first_page=1, last_page=1)
            console_log("pdf2image conversion test succeeded (Poppler working).", "OK")
        except Exception as e:
            console_log(f"pdf2image conversion test failed: {e}", "ERROR")
    else:
        console_log("Skipping pdf2image test because Poppler not found.", "WARN")
else:
    console_log(f"No sample PDF at {sample_pdf}; skipping conversion test.", "INFO")

console_log("Diagnostics complete.", "OK")


[2026-04-14 07:49:26] INFO: Running diagnostics: checking system OCR dependencies...
[2026-04-14 07:50:18] OK: Tesseract found: C:\Users\SMANYEL\AppData\Local\Programs\Tesseract-OCR\tesseract.EXE - version 5.5.0.20241111
[2026-04-14 07:50:18] OK: Poppler utility found: C:\Program Files\poppler-25.12.0\Library\bin\pdftoppm.EXE
[2026-04-14 07:50:32] OK: pdf2image conversion test succeeded (Poppler working).
[2026-04-14 07:50:32] OK: Diagnostics complete.


In [ ]:
# Step 2: Configure IBM Granite & Security Guardrails
import os
from getpass import getpass
import certifi
from llama_index.core import Settings
from llama_index.llms.replicate import Replicate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


# Networking hardening for enterprise SSL + HF transport
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
os.environ.setdefault("CURL_CA_BUNDLE", certifi.where())

# On Windows/corporate networks, this merges Windows cert store into certifi trust
try:
    import certifi_win32  # noqa: F401
    print("✅ Windows certificate store bridge enabled (certifi-win32).")
except Exception as e:
    print(f"⚠️ certifi-win32 not active ({e}); using certifi defaults.")

# Enter your REPLICATE_API_KEY safely (env var first, then prompt)
replicate_token = os.getenv("REPLICATE_API_TOKEN", "").strip()
if not replicate_token:
    try:
        replicate_token = getpass("Enter REPLICATE_API_TOKEN: ").strip()
    except Exception:
        replicate_token = input("Enter REPLICATE_API_TOKEN: ").strip()

if not replicate_token:
    raise ValueError("REPLICATE_API_TOKEN is required to continue.")

os.environ["REPLICATE_API_TOKEN"] = replicate_token

# ACADEMIC SHIELD FIX: Granite 3.1 with Extended Patience
llm = Replicate(
    model="ibm-granite/granite-3.1-8b-instruct",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    request_timeout=600.0,  # Increased to 10 minutes for complex Statistics/Leadership PDFs
    system_prompt=(
        "You are an academic research assistant. "
        "Answer ONLY using the evidence provided in the user's message and in the provided context. "
        "Do NOT introduce external knowledge or assumptions not present in the provided text. "
        "If the answer cannot be found in the provided evidence, state exactly what is missing. "
        "Write in formal academic prose. Produce thorough, well-structured essays "
        "that are fully grounded in the supplied source material."
    )
)

# Embedding model with robust fallback path
try:
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    print("✅ Primary embedding model loaded: BAAI/bge-small-en-v1.5")
except Exception as e1:
    print(f"⚠️ Primary embedding load failed: {e1}")
    try:
        embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
        print("✅ Fallback embedding model loaded: all-MiniLM-L6-v2")
    except Exception as e2:
        from llama_index.core.embeddings import MockEmbedding
        embed_model = MockEmbedding(embed_dim=384)
        print(f"⚠️ HF downloads unavailable; using MockEmbedding fallback ({e2}).")

Settings.llm = llm
Settings.embed_model = embed_model

print("🚀 Granite 3.1 Ready with Academic Shield (Timeout: 600s | Essay Mode)")


✅ Windows certificate store bridge enabled (certifi-win32).
[2026-04-14 08:56:15] INFO: 08:56:15 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


[2026-04-14 08:56:17] INFO: 08:56:17 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-14 08:56:17] WARNING: 08:56:17 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-04-14 08:56:19] INFO: 08:56:19 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
[2026-04-14 08:56:20] INFO: 08:56:20 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-14 08:56:20] INFO: 08:56:20 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
[2026-04-1

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-04-14 08:56:32] INFO: 08:56:32 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-14 08:56:33] INFO: 08:56:33 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
[2026-04-14 08:56:34] INFO: 08:56:34 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-14 08:56:34] INFO: 08:56:34 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
[2026-04-14 08:56:36] INFO: 08:56:36 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
[2026-04-14 08:

In [7]:
# Step 3: Automated PDF Ingestion (Google Drive)
import os
import re
import requests

def extract_drive_file_id(drive_url: str) -> str:
    """Extract Google Drive file id from common share URL formats."""
    patterns = [
        r"/d/([A-Za-z0-9_-]+)",
        r"[?&]id=([A-Za-z0-9_-]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, drive_url)
        if match:
            return match.group(1)
    raise ValueError("Could not extract a Google Drive file id from the provided link.")

def _raise_drive_access_error(file_id: str, status_code: int | None = None) -> None:
    status_msg = f"HTTP {status_code}. " if status_code else ""
    raise PermissionError(
        status_msg
        + "Google Drive blocked direct download for this file. "
        + "Set sharing to 'Anyone with the link: Viewer' and retry. "
        + f"You can also test with: https://drive.google.com/uc?export=download&id={file_id}"
    )

def download_pdf_from_drive(drive_url: str, save_path: str, session: requests.Session | None = None) -> None:
    """Download a Drive file and ensure the saved artifact is a valid PDF."""
    session = session or requests.Session()
    file_id = extract_drive_file_id(drive_url)

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        ),
        "Accept": "text/html,application/pdf,application/octet-stream,*/*",
        "Referer": "https://drive.google.com/",
    }

    endpoints = [
        ("https://drive.google.com/uc", {"export": "download", "id": file_id}),
        ("https://drive.usercontent.google.com/download", {"id": file_id, "export": "download"}),
    ]

    last_status = None
    last_error = None
    for base_url, params in endpoints:
        response = session.get(
            base_url,
            params=params,
            headers=headers,
            stream=True,
            allow_redirects=True,
            timeout=60,
        )
        last_status = response.status_code

        # Drive can require a confirm token for large files.
        token = None
        for key, val in response.cookies.items():
            if key.startswith("download_warning"):
                token = val
                break

        # Only scan for a confirm token when the response is HTML;
        # reading .content on a streaming PDF response would consume the stream.
        if not token and response.status_code < 400:
            content_type = response.headers.get("Content-Type", "").lower()
            if "text/html" in content_type:
                preview_html = response.content.decode("utf-8", errors="ignore")
                token_match = re.search(r"confirm=([0-9A-Za-z-_]+)", preview_html)
                if token_match:
                    token = token_match.group(1)

        if token:
            params = dict(params)
            params["confirm"] = token
            response = session.get(
                base_url,
                params=params,
                headers=headers,
                stream=True,
                allow_redirects=True,
                timeout=60,
            )
            last_status = response.status_code

        # If Drive still denies access, try next endpoint.
        if response.status_code in (401, 403):
            continue

        response.raise_for_status()

        with open(save_path, "wb") as file_obj:
            for chunk in response.iter_content(chunk_size=32768):
                if chunk:
                    file_obj.write(chunk)

        with open(save_path, "rb") as file_obj:
            header = file_obj.read(5)

        if header == b"%PDF-":
            console_log(f"Document secured: {save_path}", "OK")
            return

        with open(save_path, "rb") as file_obj:
            preview = file_obj.read(400).decode("utf-8", errors="ignore")
        os.remove(save_path)
        # Store the error and try the next endpoint instead of raising immediately.
        last_error = ValueError(
            "Downloaded file is not a valid PDF. "
            "Google Drive likely returned an HTML page (permissions/login/interstitial). "
            "Set sharing to 'Anyone with the link: Viewer' and retry. "
            f"Preview: {preview[:120]!r}"
        )
        continue

    if last_error is not None:
        raise last_error
    _raise_drive_access_error(file_id=file_id, status_code=last_status)

def extract_text_or_ocr(pdf_path: str, dpi: int = 300) -> str:
    """Use PyMuPDF extraction first; fall back to OCR and return chosen source file path."""
    try:
        import fitz  # PyMuPDF
    except Exception:
        fitz = None
        console_log("PyMuPDF not available; OCR fallback may be required.", "WARN")

    extracted_text = ""
    if fitz is not None:
        try:
            with fitz.open(pdf_path) as doc:
                for page in doc:
                    extracted_text += page.get_text() + "\n"
        except Exception as exc:
            console_log(f"PyMuPDF extraction error: {exc}", "WARN")
            extracted_text = ""

    if len(extracted_text.strip()) > 50:
        console_log("PDF contains extractable text (PyMuPDF).", "OK")
        return pdf_path

    console_log("No reliable extractable text found; using OCR fallback.", "WARN")
    try:
        from pdf2image import convert_from_path
        import pytesseract
    except Exception:
        console_log(
            "Missing OCR packages/dependencies (pdf2image, pytesseract, Poppler, Tesseract).",
            "ERROR",
        )
        return pdf_path

    # Reuse notebook-level resolver so OCR works even when PATH is missing
    tess_cmd = resolve_tesseract_cmd() if "resolve_tesseract_cmd" in globals() else None
    if tess_cmd:
        console_log(f"Using Tesseract executable: {tess_cmd}", "INFO")

    try:
        _ = pytesseract.get_tesseract_version()
    except Exception:
        console_log(
            "Tesseract executable not found. Install it and rerun Step 3.",
            "ERROR",
        )
        return pdf_path

    try:
        images = convert_from_path(pdf_path, dpi=dpi)
        ocr_text = ""
        for image in images:
            ocr_text += pytesseract.image_to_string(image) + "\n"

        txt_path = pdf_path + ".ocr.txt"
        with open(txt_path, "w", encoding="utf-8") as file_obj:
            file_obj.write(ocr_text)

        console_log(f"OCR complete: {txt_path}", "OK")
        return txt_path
    except Exception as exc:
        console_log(f"OCR failed: {exc}", "ERROR")
        return pdf_path

drive_link = input("📌 Paste Google Drive Link: ").strip()
DATA_DIR = "academic_data"
os.makedirs(DATA_DIR, exist_ok=True)

pdf_path = os.path.join(DATA_DIR, "source_material.pdf")
download_pdf_from_drive(drive_link, pdf_path)

# source_file is consumed by Step 4 and can be either PDF or OCR text file
source_file = extract_text_or_ocr(pdf_path)
console_log(f"Using source file: {source_file}", "OK")


[2026-04-14 09:39:49] OK: Document secured: academic_data\source_material.pdf
[2026-04-14 09:39:50] OK: PDF contains extractable text (PyMuPDF).
[2026-04-14 09:39:50] OK: Using source file: academic_data\source_material.pdf


In [8]:
# Step 4: Semantic Chunking + Memory Palace Metadata
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.node_parser import SemanticSplitterNodeParser

# 'source_file' is produced by the previous cell and may be a PDF or a .ocr.txt file
documents = SimpleDirectoryReader(input_files=[source_file]).load_data()

file_name = os.path.basename(source_file)
default_wing = "Projects"
default_room = os.path.splitext(file_name)[0].replace(" ", "_") or "General_Topic"
default_hall = "Research_Evidence"
default_drawer_type = "Raw_Text" if source_file.endswith(".txt") else "PDF_Text"

wing_label = input(f"🪽 Wing label [{default_wing}]: ").strip() or default_wing
room_label = input(f"🚪 Room label [{default_room}]: ").strip() or default_room
hall_label = input(f"🏛️ Hall label [{default_hall}]: ").strip() or default_hall
drawer_type = input(f"🗄️ Drawer type [{default_drawer_type}]: ").strip() or default_drawer_type

# Stable doc ids are required for true UPSERT behavior across reruns.
for i, doc in enumerate(documents):
    doc.doc_id = f"{file_name}::doc::{i}"
    doc.metadata["source"] = file_name
    doc.metadata["file_name"] = file_name
    doc.metadata["wing"] = wing_label
    doc.metadata["room"] = room_label
    doc.metadata["hall"] = hall_label
    doc.metadata["drawer_type"] = drawer_type

splitter = SemanticSplitterNodeParser(
    buffer_size=3,
    breakpoint_percentile_threshold=95,
    embed_model=embed_model,
)

nodes = splitter.get_nodes_from_documents(documents)
for n in nodes:
    n.metadata["source"] = file_name
    n.metadata["file_name"] = file_name
    n.metadata["wing"] = wing_label
    n.metadata["room"] = room_label
    n.metadata["hall"] = hall_label
    n.metadata["drawer_type"] = drawer_type

# A compact current-doc engine is used for traceable citations in Step 6.
current_doc_index = VectorStoreIndex(nodes, embed_model=embed_model)
current_doc_query_engine = current_doc_index.as_query_engine(
    similarity_top_k=5,
    response_mode="compact",
    streaming=False,
)

console_log(
    (
        f"Created {len(nodes)} semantic nodes from {file_name} "
        f"in wing='{wing_label}', room='{room_label}', hall='{hall_label}', drawer='{drawer_type}'."
    ),
    "OK",
)

[2026-04-14 09:41:42] OK: Created 38 semantic nodes from source_material.pdf in wing='Leadership', room='Adaptive Leadership', hall='Adaptive Leadership', drawer='Adaptive Leadership'.


In [9]:
# Step 4.5: MemPalace Bridge (Persistent Long-Term Memory with UPSERTS + Verification)
import os
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline, DocstoreStrategy
from llama_index.core.storage.docstore import SimpleDocumentStore

PALACE_DB_PATH = os.getenv("PALACE_DB_PATH", "./mempalace_db")
PALACE_COLLECTION = os.getenv("PALACE_COLLECTION", "academic_palace")
PALACE_DOCSTORE_PATH = os.path.join(PALACE_DB_PATH, "docstore.json")

palace_client = chromadb.PersistentClient(path=PALACE_DB_PATH)
palace_collection = palace_client.get_or_create_collection(
    PALACE_COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

palace_vector_store = ChromaVectorStore(chroma_collection=palace_collection)
if os.path.exists(PALACE_DOCSTORE_PATH):
    palace_docstore = SimpleDocumentStore.from_persist_path(PALACE_DOCSTORE_PATH)
else:
    palace_docstore = SimpleDocumentStore()

def _build_chroma_payload(nodes_for_payload):
    payload_ids = []
    payload_docs = []
    payload_metas = []
    payload_embeddings = []

    for idx, node in enumerate(nodes_for_payload):
        try:
            node_text = node.get_content(metadata_mode="none")
        except Exception:
            node_text = node.get_content() if hasattr(node, "get_content") else ""

        if not node_text or not node_text.strip():
            continue

        meta = dict(getattr(node, "metadata", {}) or {})
        meta["source"] = meta.get("source", os.path.basename(source_file))
        meta["file_name"] = meta.get("file_name", os.path.basename(source_file))
        meta["wing"] = meta.get("wing", globals().get("wing_label", "Projects"))
        meta["room"] = meta.get("room", globals().get("room_label", "General_Topic"))
        meta["hall"] = meta.get("hall", globals().get("hall_label", "Research_Evidence"))
        meta["drawer_type"] = meta.get("drawer_type", globals().get("drawer_type", "PDF_Text"))

        node_id = getattr(node, "node_id", None) or getattr(node, "id_", None) or f"{os.path.basename(source_file)}::node::{idx}"
        embedding = getattr(node, "embedding", None)
        if embedding is None:
            embedding = embed_model.get_text_embedding(node_text)

        payload_ids.append(str(node_id))
        payload_docs.append(node_text)
        payload_metas.append(meta)
        payload_embeddings.append(embedding)

    return payload_ids, payload_docs, payload_metas, payload_embeddings

# Track collection size before ingestion to verify persistence behavior.
before_count = palace_collection.count()

# UPSERT strategy prevents duplicate vectors when the same source is ingested again.
palace_pipeline = IngestionPipeline(
    transformations=[splitter],
    docstore=palace_docstore,
    vector_store=palace_vector_store,
    docstore_strategy=DocstoreStrategy.UPSERTS,
)

pipeline_nodes = palace_pipeline.run(documents=documents)
for n in pipeline_nodes:
    n.metadata["source"] = n.metadata.get("source", os.path.basename(source_file))
    n.metadata["file_name"] = n.metadata.get("file_name", os.path.basename(source_file))
    n.metadata["wing"] = n.metadata.get("wing", globals().get("wing_label", "Projects"))
    n.metadata["room"] = n.metadata.get("room", globals().get("room_label", "General_Topic"))
    n.metadata["hall"] = n.metadata.get("hall", globals().get("hall_label", "Research_Evidence"))
    n.metadata["drawer_type"] = n.metadata.get("drawer_type", globals().get("drawer_type", "PDF_Text"))

# Persist docstore so UPSERT state survives kernel restarts.
os.makedirs(PALACE_DB_PATH, exist_ok=True)
palace_docstore.persist(persist_path=PALACE_DOCSTORE_PATH)

after_pipeline_count = palace_collection.count()
fallback_used = False
fallback_mode = "none"
fallback_upserts = 0

# Case A: pipeline produced nodes but vector count did not move -> force explicit upsert.
if pipeline_nodes and after_pipeline_count <= before_count:
    f_ids, f_docs, f_metas, f_embeddings = _build_chroma_payload(pipeline_nodes)
    if f_ids:
        palace_collection.upsert(
            ids=f_ids,
            documents=f_docs,
            metadatas=f_metas,
            embeddings=f_embeddings,
        )
        fallback_used = True
        fallback_mode = "pipeline-nodes"
        fallback_upserts = len(f_ids)

# Case B: pipeline produced zero nodes while collection is still empty -> bootstrap from Step 4 semantic nodes.
if not fallback_used and not pipeline_nodes and after_pipeline_count == 0 and globals().get("nodes"):
    b_ids, b_docs, b_metas, b_embeddings = _build_chroma_payload(globals()["nodes"])
    if b_ids:
        palace_collection.upsert(
            ids=b_ids,
            documents=b_docs,
            metadatas=b_metas,
            embeddings=b_embeddings,
        )
        fallback_used = True
        fallback_mode = "bootstrap-step4-nodes"
        fallback_upserts = len(b_ids)

after_final_count = palace_collection.count()

# Re-open an index from the vector store so retrieval can span all prior runs.
palace_index = VectorStoreIndex.from_vector_store(
    palace_vector_store,
    embed_model=embed_model,
)

palace_query_engine = palace_index.as_query_engine(
    similarity_top_k=8,
    response_mode="compact",
    streaming=False,
)

console_log(
    f"MemPalace connected at '{PALACE_DB_PATH}' (collection: '{PALACE_COLLECTION}').",
    "OK",
)
console_log(
    (
        f"UPSERT ingestion complete: {len(pipeline_nodes)} nodes processed "
        f"for wing='{globals().get('wing_label', 'Projects')}', "
        f"room='{globals().get('room_label', 'General_Topic')}', "
        f"hall='{globals().get('hall_label', 'Research_Evidence')}', "
        f"drawer='{globals().get('drawer_type', 'PDF_Text')}'."
    ),
    "OK",
)
console_log(
    f"Palace count check: before={before_count}, after_pipeline={after_pipeline_count}, final={after_final_count}",
    "INFO",
)
if fallback_used:
    console_log(
        f"Fallback '{fallback_mode}' upsert executed for {fallback_upserts} nodes.",
        "WARN",
    )

[2026-04-14 09:43:10] OK: MemPalace connected at './mempalace_db' (collection: 'academic_palace').
[2026-04-14 09:43:10] OK: UPSERT ingestion complete: 38 nodes processed for wing='Leadership', room='Adaptive Leadership', hall='Adaptive Leadership', drawer='Adaptive Leadership'.
[2026-04-14 09:43:10] INFO: Palace count check: before=713, after_pipeline=713, final=751
[2026-04-14 09:43:10] WARN: Fallback 'pipeline-nodes' upsert executed for 38 nodes.


In [25]:
# Step 4.6: Memory Stats Dashboard (Hierarchy Expanded + Lightweight Scan)
from collections import Counter
import os

palace_count = palace_collection.count()
print(f"🏰 Palace Stats: {palace_count} semantic memories stored.")
print(f"📂 Current Collection: {palace_collection.name}")

# Use a capped metadata scan by default to avoid dashboard bottlenecks on large collections.
DASHBOARD_MAX_SCAN = int(os.getenv("PALACE_DASHBOARD_MAX_SCAN", "1500"))
DASHBOARD_PREVIEW_LIMIT = int(os.getenv("PALACE_DASHBOARD_PREVIEW_LIMIT", "8"))

try:
    scan_limit = min(palace_count, DASHBOARD_MAX_SCAN) if palace_count > 0 else 0
    all_metadatas = []

    if scan_limit > 0:
        all_meta_payload = palace_collection.get(limit=scan_limit, include=["metadatas"])
        if isinstance(all_meta_payload, dict):
            all_metadatas = all_meta_payload.get("metadatas", []) or []

    wing_counter = Counter((m or {}).get("wing", "Projects") for m in all_metadatas)
    room_counter = Counter((m or {}).get("room", "General_Topic") for m in all_metadatas)
    hall_counter = Counter((m or {}).get("hall", "Research_Evidence") for m in all_metadatas)
    drawer_counter = Counter((m or {}).get("drawer_type", "PDF_Text") for m in all_metadatas)

    if palace_count > scan_limit:
        print(
            f"ℹ️ Dashboard scanned {scan_limit}/{palace_count} memories (set PALACE_DASHBOARD_MAX_SCAN for deeper scans)."
        )

    if wing_counter:
        print("\n🪽 Wing distribution:")
        for wing, cnt in wing_counter.most_common():
            print(f"  - {wing}: {cnt}")

    if room_counter:
        print("\n🚪 Top rooms:")
        for room, cnt in room_counter.most_common(10):
            print(f"  - {room}: {cnt}")

    if hall_counter:
        print("\n🏛️ Hall distribution:")
        for hall, cnt in hall_counter.most_common():
            print(f"  - {hall}: {cnt}")

    if drawer_counter:
        print("\n🗄️ Drawer types:")
        for drawer, cnt in drawer_counter.most_common():
            print(f"  - {drawer}: {cnt}")

    preview = palace_collection.peek(limit=DASHBOARD_PREVIEW_LIMIT)
    metadatas = preview.get("metadatas", []) if isinstance(preview, dict) else []
    if metadatas:
        print("\n🧾 Sample memories:")
        for i, meta in enumerate(metadatas[:DASHBOARD_PREVIEW_LIMIT], start=1):
            m = meta or {}
            print(
                f"  {i}. file={m.get('file_name', 'Unknown')} | "
                f"wing={m.get('wing', 'Projects')} | "
                f"room={m.get('room', 'General_Topic')} | "
                f"hall={m.get('hall', 'Research_Evidence')} | "
                f"drawer={m.get('drawer_type', 'PDF_Text')}"
            )
except Exception as e:
    print(f"⚠️ Could not preview memory metadata: {e}")

🏰 Palace Stats: 713 semantic memories stored.
📂 Current Collection: academic_palace

🪽 Wing distribution:
  - Projects: 359
  - Leadership: 354

🚪 Top rooms:
  - General_Topic: 359
  - Adaptive Leadership: 354

🏛️ Hall distribution:
  - Adaptive Leadership: 713

🗄️ Drawer types:
  - PDF_Text: 359
  - Adaptive Leadership: 354

🧾 Sample memories:
  1. file=source_material.pdf | wing=Projects | room=General_Topic | hall=Adaptive Leadership | drawer=PDF_Text
  2. file=source_material.pdf | wing=Projects | room=General_Topic | hall=Adaptive Leadership | drawer=PDF_Text
  3. file=source_material.pdf | wing=Projects | room=General_Topic | hall=Adaptive Leadership | drawer=PDF_Text
  4. file=source_material.pdf | wing=Projects | room=General_Topic | hall=Adaptive Leadership | drawer=PDF_Text
  5. file=source_material.pdf | wing=Projects | room=General_Topic | hall=Adaptive Leadership | drawer=PDF_Text
  6. file=source_material.pdf | wing=Projects | room=General_Topic | hall=Adaptive Leadership

In [ ]:
# Step 4.6b: Label Index Retriever — inspect all wing/room/hall labels + retrieve stored chunks
from collections import defaultdict
import os

# ── 1. Build the full label index from ChromaDB ─────────────────────────────
def get_label_index(collection, max_scan: int = 5000) -> dict:
    """Return all unique (wing, room, hall, file_name) combinations stored in the palace."""
    count = collection.count()
    if count == 0:
        print("⚠️ Palace is empty. Run Steps 4 and 4.5 first.")
        return {}

    limit = min(count, max_scan)
    payload = collection.get(limit=limit, include=["metadatas"])
    metadatas = (payload or {}).get("metadatas", []) or []

    index = defaultdict(set)
    for m in metadatas:
        m = m or {}
        wing  = m.get("wing",  "Projects")
        room  = m.get("room",  "General_Topic")
        hall  = m.get("hall",  "Research_Evidence")
        fname = m.get("file_name", "Unknown")
        index[(wing, room, hall)].add(fname)

    return dict(index)

label_index = get_label_index(palace_collection)

print(f"\n📚 LABEL INDEX — {len(label_index)} unique wing/room/hall combination(s) in palace:\n")
for (wing, room, hall), files in sorted(label_index.items()):
    print(f"  Wing: {wing}  |  Room: {room}  |  Hall: {hall}")
    for f in sorted(files):
        print(f"    └─ {f}")

# ── 2. Retrieve stored chunks by label ──────────────────────────────────────
def retrieve_by_label(
    wing: str | None = None,
    room: str | None = None,
    hall: str | None = None,
    query: str = "",
    top_k: int = 5,
    show_text: bool = True,
) -> list[dict]:
    """
    Retrieve stored chunks from the palace filtered by label(s).
    Optionally run a semantic similarity query on top of the filter.

    Args:
        wing:      Filter by wing label (None = no filter).
        room:      Filter by room label (None = no filter).
        hall:      Filter by hall label (None = no filter).
        query:     Semantic search string. Leave empty to return by metadata only.
        top_k:     Number of results to return.
        show_text: Print results to output.

    Returns:
        List of result dicts with keys: file_name, wing, room, hall, drawer_type, text.
    """
    from llama_index.core.vector_stores import MetadataFilter, MetadataFilters

    active_filters = []
    if wing:
        active_filters.append(MetadataFilter(key="wing", value=wing))
    if room:
        active_filters.append(MetadataFilter(key="room", value=room))
    if hall:
        active_filters.append(MetadataFilter(key="hall", value=hall))

    retriever_kwargs = {"similarity_top_k": top_k}
    if active_filters:
        retriever_kwargs["filters"] = MetadataFilters(filters=active_filters)

    retriever = palace_index.as_retriever(**retriever_kwargs)

    search_query = query.strip() or (
        f"{wing or ''} {room or ''} {hall or ''}".strip() or "general"
    )
    nodes = retriever.retrieve(search_query)

    results = []
    for i, item in enumerate(nodes, start=1):
        node = getattr(item, "node", item)
        meta = getattr(node, "metadata", {}) or {}
        try:
            text = node.get_content().strip()
        except Exception:
            text = ""

        result = {
            "file_name":   meta.get("file_name", "Unknown"),
            "wing":        meta.get("wing",  "Projects"),
            "room":        meta.get("room",  "General_Topic"),
            "hall":        meta.get("hall",  "Research_Evidence"),
            "drawer_type": meta.get("drawer_type", "PDF_Text"),
            "text":        text,
        }
        results.append(result)

        if show_text:
            print(f"\n── Result {i} ──────────────────────────────────────")
            print(f"  File:   {result['file_name']}")
            print(f"  Labels: wing={result['wing']} | room={result['room']} | hall={result['hall']} | drawer={result['drawer_type']}")
            print(f"  Text:   {text[:400]}{'...' if len(text) > 400 else ''}")

    if show_text and not results:
        print("⚠️ No chunks found for the given label combination.")

    return results


# ── 3. Usage examples (edit labels to match your ingested PDFs) ─────────────
print("\n" + "=" * 55)
print("retrieve_by_label() is ready. Example usage:")
print("=" * 55)
print("""
# Retrieve all chunks from a specific room:
retrieve_by_label(room="Adaptive_Leadership", top_k=5)

# Retrieve with a semantic query inside a wing:
retrieve_by_label(wing="Projects", query="What is adaptive leadership?", top_k=3)

# Retrieve by full path:
retrieve_by_label(wing="Projects", room="Adaptive_Leadership", hall="Theory", top_k=5)

# Retrieve without printing (for programmatic use):
chunks = retrieve_by_label(room="Case_Studies", show_text=False)
""")


In [10]:
# Step 4.7: Load local awesome-notebookLM style packs
import os
import re
from pathlib import Path

NOTEBOOKLM_REPO_PATH = Path(os.getenv("NOTEBOOKLM_REPO_PATH", r"C:\Users\SMANYEL\awesome-notebookLM"))
NOTEBOOKLM_README_PATH = NOTEBOOKLM_REPO_PATH / "README.md"

def _normalize_heading(text: str) -> str:
    return text.strip().strip("*").strip(":").lower()

def _extract_style_blocks(markdown_text: str) -> dict:
    """Extract style sections and their first fenced prompt blocks from awesome-notebookLM README."""
    section_pattern = re.compile(r"^##\s+(.+?)\n(.*?)(?=^##\s+|\Z)", re.MULTILINE | re.DOTALL)
    fence_pattern = re.compile(r"```\n(.*?)```", re.DOTALL)

    style_blocks = {}
    for heading, body in section_pattern.findall(markdown_text):
        heading_clean = _normalize_heading(heading)
        code_blocks = fence_pattern.findall(body)
        if code_blocks:
            style_blocks[heading_clean] = code_blocks[0].strip()
    return style_blocks

def _pick_style(extracted: dict, *keywords: str) -> str:
    for key, val in extracted.items():
        if all(kw in key for kw in keywords):
            return val
    return ""

def load_notebooklm_style_library(readme_path: Path = NOTEBOOKLM_README_PATH) -> dict:
    if not readme_path.exists():
        raise FileNotFoundError(f"awesome-notebookLM README not found at: {readme_path}")

    markdown_text = readme_path.read_text(encoding="utf-8", errors="ignore")
    extracted = _extract_style_blocks(markdown_text)

    # Curated aliases for Academic Truth Engine usage.
    curated = {
        "editorial": _pick_style(extracted, "modern", "newspaper"),
        "minimal": _pick_style(extracted, "sharp-edged", "minimalism"),
        "magazine": _pick_style(extracted, "magazine"),
        "neon-tech": _pick_style(extracted, "tech", "neon"),
        "digital-pop": _pick_style(extracted, "digital", "pop"),
        "artifact": _pick_style(extracted, "anti-gravity", "artifact"),
    }

    return {
        "raw_sections": extracted,
        "curated": {k: v for k, v in curated.items() if v},
        "readme_path": str(readme_path),
    }

style_library = load_notebooklm_style_library()
console_log(
    f"Loaded awesome-notebookLM styles from {style_library['readme_path']}. "
    f"Curated styles: {', '.join(style_library['curated'].keys())}",
    "OK",
)

[2026-04-14 09:44:12] OK: Loaded awesome-notebookLM styles from C:\Users\SMANYEL\awesome-notebookLM\README.md. Curated styles: editorial, minimal, magazine, neon-tech, digital-pop, artifact


In [11]:
# Step 4.8: Visual Answer Renderer + NotebookLM Slide Brief Export
import json
from datetime import datetime
from IPython.display import HTML, display

def render_answer_panel(question: str, result: dict, max_sources: int = 8):
    """Render a clean, dashboard-like answer panel in notebook output."""
    answer = (result or {}).get("answer", "")
    retrieval_mode = (result or {}).get("retrieval_mode", "unknown")
    sources = (result or {}).get("sources", [])[:max_sources]

    source_items = []
    for src in sources:
        source_items.append(
            "<li><b>{file}</b> | wing={wing} | room={room} | hall={hall} | drawer={drawer}<br>"
            "<span style='color:#444'>{snippet}</span></li>".format(
                file=src.get("file_name", "Unknown"),
                wing=src.get("wing", "Projects"),
                room=src.get("room", "General_Topic"),
                hall=src.get("hall", "Research_Evidence"),
                drawer=src.get("drawer_type", "PDF_Text"),
                snippet=(src.get("snippet", "") or "No snippet available"),
            )
        )

    sources_html = "".join(source_items) if source_items else "<li>No sources returned.</li>"

    panel = f"""
    <div style='font-family: Segoe UI, Arial, sans-serif; border:1px solid #d9d9d9; border-radius:14px; padding:16px; background:#fafafa;'>
      <div style='font-size:12px; color:#666; margin-bottom:6px;'>ACADEMIC TRUTH ENGINE V3</div>
      <div style='font-size:20px; font-weight:700; margin-bottom:8px;'>Evidence-first Answer</div>
      <div style='font-size:13px; color:#333; margin-bottom:10px;'><b>Question:</b> {question}</div>
      <div style='font-size:13px; color:#333; margin-bottom:10px;'><b>Retrieval Mode:</b> {retrieval_mode}</div>
      <div style='padding:10px; border-radius:10px; background:white; border:1px solid #ececec; white-space:pre-wrap; line-height:1.45;'>{answer}</div>
      <div style='margin-top:12px; font-size:14px; font-weight:700;'>Sources</div>
      <ol style='margin-top:6px; padding-left:18px; line-height:1.4;'>{sources_html}</ol>
    </div>
    """
    display(HTML(panel))

def build_slide_brief(question: str, result: dict, style_key: str = "artifact") -> dict:
    """Create a structured brief using local awesome-notebookLM style templates."""
    curated = (style_library or {}).get("curated", {})
    style_prompt = curated.get(style_key, "")
    sources = (result or {}).get("sources", [])

    source_summary = []
    for src in sources[:10]:
        source_summary.append({
            "file": src.get("file_name", "Unknown"),
            "wing": src.get("wing", "Projects"),
            "room": src.get("room", "General_Topic"),
            "hall": src.get("hall", "Research_Evidence"),
            "drawer": src.get("drawer_type", "PDF_Text"),
            "snippet": src.get("snippet", "")
        })

    brief = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "question": question,
        "answer": (result or {}).get("answer", ""),
        "retrieval_mode": (result or {}).get("retrieval_mode", "unknown"),
        "style_key": style_key,
        "style_prompt": style_prompt,
        "source_summary": source_summary,
        "instructions": [
            "Use source_summary as the only evidence base.",
            "One slide should carry one claim.",
            "Preserve strict citation traceability to source snippets.",
            "Avoid unsupported synthesis beyond provided evidence."
        ]
    }
    return brief

def export_slide_brief(brief: dict, output_dir: str = "academic_data/visual_briefs") -> str:
    os.makedirs(output_dir, exist_ok=True)
    safe_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = os.path.join(output_dir, f"slide_brief_{safe_ts}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(brief, f, ensure_ascii=False, indent=2)
    console_log(f"Slide brief exported: {out_path}", "OK")
    return out_path

print("✅ Visual helpers ready:")
print("- render_answer_panel(question, result)")
print("- build_slide_brief(question, result, style_key='artifact')")
print("- export_slide_brief(brief)")

✅ Visual helpers ready:
- render_answer_panel(question, result)
- build_slide_brief(question, result, style_key='artifact')
- export_slide_brief(brief)


In [12]:
# Step 5: Advanced Query Fusion (Sequential Stability Mode)
import os
import sys
from pathlib import Path
import importlib.util

# 1. Clean up local path references
local_pack_root = Path("query_rewriting_pack").resolve()
if str(local_pack_root) not in sys.path:
    sys.path.insert(0, str(local_pack_root))

# 2. Robust Import Logic
try:
    from llama_index.packs.fusion_retriever.query_rewrite.base import QueryRewritingRetrieverPack
except Exception:
    # Fallback to direct file loading if the namespace is messy
    base_file = local_pack_root / "llama_index" / "packs" / "fusion_retriever" / "query_rewrite" / "base.py"
    spec = importlib.util.spec_from_file_location("local_query_rewrite_base", str(base_file))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    QueryRewritingRetrieverPack = module.QueryRewritingRetrieverPack

# 3. Sequential stability for the current document retriever
query_rewriting_pack = QueryRewritingRetrieverPack(
    nodes,
    chunk_size=256,
    vector_similarity_top_k=5,
    num_queries=1,
)

print("🚀 Search Engine set to SINGLE-STREAM mode.")
print("✅ Stability confirmed: Sequential processing enabled.")
console_log("Advanced Query Fusion Engine Ready (Sequential Fix)", "OK")
console_log("Hybrid mode ready: short-term fusion + long-term MemPalace retrieval.", "OK")

resource module not available on Windows
🚀 Search Engine set to SINGLE-STREAM mode.
✅ Stability confirmed: Sequential processing enabled.
[2026-04-14 09:45:39] OK: Advanced Query Fusion Engine Ready (Sequential Fix)
[2026-04-14 09:45:39] OK: Hybrid mode ready: short-term fusion + long-term MemPalace retrieval.


In [13]:
# Step 5.5: Runtime preset switcher (run this before Step 6)
import os

# One-line choice: 'visual_only' | 'visual_brief' | 'visual_brief_export'
ATE_PRESET = "visual_brief"

PRESETS = {
    "visual_only": {
        "ATE_VISUAL_PANEL": "1",
        "ATE_AUTO_BRIEF": "0",
        "ATE_AUTO_BRIEF_EXPORT": "0",
    },
    "visual_brief": {
        "ATE_VISUAL_PANEL": "1",
        "ATE_AUTO_BRIEF": "1",
        "ATE_AUTO_BRIEF_EXPORT": "0",
    },
    "visual_brief_export": {
        "ATE_VISUAL_PANEL": "1",
        "ATE_AUTO_BRIEF": "1",
        "ATE_AUTO_BRIEF_EXPORT": "1",
    },
}

if ATE_PRESET not in PRESETS:
    raise ValueError(f"Unknown ATE_PRESET='{ATE_PRESET}'. Choose one of: {', '.join(PRESETS)}")

for key, value in PRESETS[ATE_PRESET].items():
    os.environ[key] = value

# Optional style override for auto brief generation in Step 6.
os.environ.setdefault("ATE_BRIEF_STYLE", "artifact")

print("✅ ATE runtime preset applied:", ATE_PRESET)
print(
    f"ATE_VISUAL_PANEL={os.getenv('ATE_VISUAL_PANEL')} | "
    f"ATE_AUTO_BRIEF={os.getenv('ATE_AUTO_BRIEF')} | "
    f"ATE_AUTO_BRIEF_EXPORT={os.getenv('ATE_AUTO_BRIEF_EXPORT')} | "
    f"ATE_BRIEF_STYLE={os.getenv('ATE_BRIEF_STYLE')}"
)

✅ ATE runtime preset applied: visual_brief
ATE_VISUAL_PANEL=1 | ATE_AUTO_BRIEF=1 | ATE_AUTO_BRIEF_EXPORT=0 | ATE_BRIEF_STYLE=artifact


In [ ]:
# Step 6: The Research Loop (Q&A) - Hierarchical Hybrid Memory + Semantic Citations
import os
import time
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters

WING_KEYWORD_MAP = {
    "Projects": ["project", "assignment", "capstone", "research", "analysis"],
    "People": ["stakeholder", "leader", "manager", "person", "team"],
}

ROOM_KEYWORD_MAP = {
    "Adaptive_Leadership": ["adaptive", "leadership", "capacity"],
    "Case_Studies": ["case study", "case", "scenario", "example"],
    "System_Stats": ["stat", "statistics", "variance", "mean", "regression", "dataset"],
    "Systems_Thinking": ["system", "systems thinking", "feedback loop", "causal"],
}

HALL_KEYWORD_MAP = {
    "Research_Evidence": ["evidence", "citation", "source", "proof"],
    "Theory": ["theory", "framework", "model", "concept"],
    "Application": ["apply", "application", "implementation", "practice"],
}

# Runtime controls to ease bottlenecks without changing architecture.
RUNTIME_MODE = os.getenv("ATE_RUNTIME_MODE", "fast").strip().lower()
if RUNTIME_MODE not in {"fast", "deep"}:
    RUNTIME_MODE = "fast"

PALACE_TOP_K = 8 if RUNTIME_MODE == "fast" else 12
MAX_FILTER_ATTEMPTS = 3 if RUNTIME_MODE == "fast" else 7

# LLM synthesis is ON by default so the engine produces full essay-style answers.
# Set ATE_ENABLE_SYNTHESIS=0 or ATE_ENABLE_NETWORK_LLM=0 to fall back to local-only mode.
ENABLE_LLM_SYNTHESIS = os.getenv("ATE_ENABLE_SYNTHESIS", "1").strip() in {"1", "true", "True"}
ENABLE_NETWORK_LLM = os.getenv("ATE_ENABLE_NETWORK_LLM", "1").strip() in {"1", "true", "True"}

# Query fusion pack can invoke network LLMs depending on implementation; keep it opt-in.
USE_FUSION_PACK = os.getenv("ATE_USE_FUSION_PACK", "0").strip() in {"1", "true", "True"}

SOURCE_PRINT_LIMIT = int(os.getenv("ATE_SOURCE_PRINT_LIMIT", "10"))
VISUAL_PANEL_ON = os.getenv("ATE_VISUAL_PANEL", "1").strip() not in {"0", "false", "False"}
AUTO_BRIEF_ON = os.getenv("ATE_AUTO_BRIEF", "0").strip() in {"1", "true", "True"}
AUTO_BRIEF_EXPORT_ON = os.getenv("ATE_AUTO_BRIEF_EXPORT", "0").strip() in {"1", "true", "True"}
BRIEF_STYLE_KEY = os.getenv("ATE_BRIEF_STYLE", "artifact").strip() or "artifact"

# Local retrievers (embedding/vector search only).
current_doc_retriever = current_doc_index.as_retriever(similarity_top_k=8)

# Cache filtered palace retrievers so repeated queries avoid reconstruction.
_palace_retriever_cache = {}

def _is_timeout_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    timeout_markers = [
        "read operation timed out",
        "timed out",
        "timeout",
        "readtimeout",
    ]
    return any(marker in msg for marker in timeout_markers)

def _to_text(obj) -> str:
    if obj is None:
        return ""
    if hasattr(obj, "response") and isinstance(obj.response, str):
        return obj.response.strip()
    if hasattr(obj, "text") and isinstance(obj.text, str):
        return obj.text.strip()
    return str(obj).strip()

def _node_obj(item):
    return getattr(item, "node", item)

def _extract_text_from_node(item) -> str:
    node = _node_obj(item)
    try:
        return node.get_content().strip()
    except Exception:
        return ""

def _nodes_to_text(items, max_items: int = 10, max_chars: int = 6000) -> str:
    snippets = []
    total = 0
    for item in items[:max_items]:
        text = _extract_text_from_node(item)
        if not text:
            continue
        text = text.replace("\n", " ").strip()
        if not text:
            continue
        remaining = max_chars - total
        if remaining <= 0:
            break
        clipped = text[:remaining]
        snippets.append(clipped)
        total += len(clipped)
    return "\n\n".join(snippets)

def _extract_sources_from_nodes(label: str, items) -> list[dict]:
    refs = []
    for item in items or []:
        node = _node_obj(item)
        meta = getattr(node, "metadata", {}) or {}
        snippet = ""
        try:
            snippet = node.get_content()[:160].replace("\n", " ").strip()
        except Exception:
            snippet = ""
        refs.append(
            {
                "retriever": label,
                "file_name": meta.get("file_name") or meta.get("source") or "Unknown",
                "wing": meta.get("wing", "Projects"),
                "room": meta.get("room", "General_Topic"),
                "hall": meta.get("hall", "Research_Evidence"),
                "drawer_type": meta.get("drawer_type", "PDF_Text"),
                "snippet": snippet,
            }
        )
    return refs

def _dedupe_sources(refs: list[dict]) -> list[dict]:
    seen = set()
    unique_refs = []
    for ref in refs:
        key = (
            ref["retriever"],
            ref["file_name"],
            ref["wing"],
            ref["room"],
            ref["hall"],
            ref["snippet"][:80],
        )
        if key in seen:
            continue
        seen.add(key)
        unique_refs.append(ref)
    return unique_refs

def _infer_by_keywords(question: str, mapping: dict[str, list[str]]) -> str | None:
    q = question.lower()
    best_label = None
    best_score = 0
    for label, keywords in mapping.items():
        score = sum(1 for kw in keywords if kw in q)
        if score > best_score:
            best_score = score
            best_label = label
    return best_label if best_score > 0 else None

def infer_hierarchy_targets(question: str) -> dict:
    return {
        "wing": _infer_by_keywords(question, WING_KEYWORD_MAP),
        "room": _infer_by_keywords(question, ROOM_KEYWORD_MAP),
        "hall": _infer_by_keywords(question, HALL_KEYWORD_MAP),
    }

def _filters_key(filters: list[MetadataFilter] | None) -> tuple:
    if not filters:
        return tuple()
    return tuple(sorted((f.key, str(f.value)) for f in filters))

def _get_palace_retriever(filters: list[MetadataFilter] | None = None):
    key = _filters_key(filters)
    if key in _palace_retriever_cache:
        return _palace_retriever_cache[key]

    kwargs = {"similarity_top_k": PALACE_TOP_K}
    if filters:
        kwargs["filters"] = MetadataFilters(filters=filters)

    retriever = palace_index.as_retriever(**kwargs)
    _palace_retriever_cache[key] = retriever
    return retriever

def _retrieve_with_filters(question: str, filters: list[MetadataFilter]):
    palace_retriever = _get_palace_retriever(filters)
    return palace_retriever.retrieve(question)

def _build_filter_candidates(targets: dict) -> list[list[MetadataFilter]]:
    filter_candidates = []
    if targets["wing"] and targets["room"] and targets["hall"]:
        filter_candidates.append([
            MetadataFilter(key="wing", value=targets["wing"]),
            MetadataFilter(key="room", value=targets["room"]),
            MetadataFilter(key="hall", value=targets["hall"]),
        ])
    if targets["wing"] and targets["room"]:
        filter_candidates.append([
            MetadataFilter(key="wing", value=targets["wing"]),
            MetadataFilter(key="room", value=targets["room"]),
        ])
    if targets["room"] and targets["hall"]:
        filter_candidates.append([
            MetadataFilter(key="room", value=targets["room"]),
            MetadataFilter(key="hall", value=targets["hall"]),
        ])
    if targets["wing"] and targets["hall"]:
        filter_candidates.append([
            MetadataFilter(key="wing", value=targets["wing"]),
            MetadataFilter(key="hall", value=targets["hall"]),
        ])
    if targets["wing"]:
        filter_candidates.append([MetadataFilter(key="wing", value=targets["wing"])])
    if targets["room"]:
        filter_candidates.append([MetadataFilter(key="room", value=targets["room"])])
    if targets["hall"]:
        filter_candidates.append([MetadataFilter(key="hall", value=targets["hall"])])
    return filter_candidates

def hierarchical_palace_retrieve(question: str):
    targets = infer_hierarchy_targets(question)
    filter_candidates = _build_filter_candidates(targets)[:MAX_FILTER_ATTEMPTS]

    for filters in filter_candidates:
        nodes = _retrieve_with_filters(question, filters)
        if nodes:
            route = ", ".join(f"{f.key}={f.value}" for f in filters)
            return nodes, f"Hierarchy-routed ({route})"

    general_nodes = _get_palace_retriever().retrieve(question)
    return general_nodes, "General palace retrieval"

def _compose_fast_answer(question: str, short_term_text: str, current_doc_text: str, palace_text: str) -> str:
    """Offline/fallback answer when LLM synthesis is disabled. Returns raw evidence for manual review."""
    combined = "\n\n".join(filter(None, [short_term_text, current_doc_text, palace_text]))
    if not combined.strip():
        return "⚠️ No evidence found in uploaded source material for this question."
    return (
        "⚠️ OFFLINE MODE — LLM synthesis disabled. Raw evidence retrieved from your PDFs:\n\n"
        + combined
        + "\n\n[Enable ATE_ENABLE_SYNTHESIS=1 and ATE_ENABLE_NETWORK_LLM=1 for full essay output.]"
    )

def run_academic_query(question, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        try:
            short_term_text = ""
            if USE_FUSION_PACK:
                try:
                    short_term_answer = query_rewriting_pack.run(question)
                    short_term_text = _to_text(short_term_answer)
                except Exception as fusion_error:
                    console_log(f"Fusion retriever unavailable: {fusion_error}", "WARN")
                    short_term_text = ""

            # Local-only retriever calls: no LLM dependency.
            current_doc_nodes = current_doc_retriever.retrieve(question)
            current_doc_text = _nodes_to_text(current_doc_nodes)

            palace_nodes, retrieval_mode_note = hierarchical_palace_retrieve(question)
            palace_text = _nodes_to_text(palace_nodes)

            if not short_term_text and not current_doc_text and not palace_text:
                return {
                    "answer": "⚠️ Information not found in source material.",
                    "sources": [],
                    "retrieval_mode": retrieval_mode_note,
                }

            if not ENABLE_LLM_SYNTHESIS or not ENABLE_NETWORK_LLM:
                final_text = _compose_fast_answer(question, short_term_text, current_doc_text, palace_text)
            else:
                synthesis_prompt = (
                    "You are an academic writing assistant. Your task is to write a thorough, "
                    "well-structured academic essay that fully answers the question below. "
                    "You MUST use ONLY the evidence provided in the source material sections. "
                    "Do NOT introduce any external knowledge, assumptions, or information not "
                    "present in the evidence. Every claim must be grounded in the provided text.\n\n"

                    "ESSAY STRUCTURE — follow this exactly:\n\n"

                    "INTRODUCTION (3 paragraphs):\n"
                    "  Paragraph 1: Define all key concepts and theories mentioned in the question, "
                    "drawing directly from the source material.\n"
                    "  Paragraph 2: Explain the academic context and importance of the topic as "
                    "evidenced in the source material.\n"
                    "  Paragraph 3: State your thesis — a precise statement of what the essay will "
                    "cover and how it answers every part of the question.\n\n"

                    "BODY (6 paragraphs):\n"
                    "  Paragraph 4: Address the first major concept or sub-question using evidence "
                    "from the source material with direct quotations or paraphrases.\n"
                    "  Paragraph 5: Address the second major concept or sub-question using evidence "
                    "from the source material.\n"
                    "  Paragraph 6: Address the third major concept or sub-question, providing "
                    "theoretical grounding from the source material.\n"
                    "  Paragraph 7: Apply the concepts to the practical/scenario context raised in "
                    "the question, supported by source material evidence.\n"
                    "  Paragraph 8: Compare, contrast, or extend the concepts using additional "
                    "evidence from the source material.\n"
                    "  Paragraph 9: Critically evaluate any limitations, gaps, or nuances found in "
                    "the source material's treatment of the topic.\n\n"

                    "CONCLUSION (3 paragraphs):\n"
                    "  Paragraph 10: Summarise the key arguments made in the body and how they "
                    "collectively answer the question.\n"
                    "  Paragraph 11: Reflect on the broader significance of the findings as "
                    "supported by the source material.\n"
                    "  Paragraph 12: Final closing statement — tie all parts of the question "
                    "together and restate the essay's core answer.\n\n"

                    "STRICT RULES:\n"
                    "- Write in formal academic prose. No bullet points or numbered lists in the essay.\n"
                    "- Each paragraph must be at least 4 sentences long.\n"
                    "- Use direct quotes or close paraphrases from the evidence where possible.\n"
                    "- After the essay, include a SOURCES section listing the file names referenced.\n"
                    "- If the evidence is insufficient to fill a paragraph, state what is missing "
                    "and work with what is available.\n\n"

                    f"QUESTION:\n{question}\n\n"
                    f"SOURCE MATERIAL — Current Document:\n{current_doc_text or 'No current-doc evidence returned.'}\n\n"
                    f"SOURCE MATERIAL — Long-Term Palace:\n{palace_text or 'No long-term palace evidence returned.'}\n\n"
                    f"SOURCE MATERIAL — Fusion Retriever:\n{short_term_text or 'Fusion pack disabled or no evidence returned.'}\n"
                )

                final_response = llm.complete(synthesis_prompt)
                final_text = _to_text(final_response) or "⚠️ Information not found in source material."

            refs = []
            refs.extend(_extract_sources_from_nodes("current-doc", current_doc_nodes))
            refs.extend(_extract_sources_from_nodes("palace", palace_nodes))
            refs = _dedupe_sources(refs)

            return {
                "answer": final_text,
                "sources": refs,
                "retrieval_mode": f"{retrieval_mode_note} | mode={RUNTIME_MODE}",
            }

        except Exception as e:
            error_msg = str(e).lower()
            if _is_timeout_error(e):
                if attempt < max_attempts:
                    wait_seconds = min(2 * attempt, 8)
                    print(
                        f"⚠️ Timeout detected (attempt {attempt}/{max_attempts}). "
                        f"Retrying in {wait_seconds}s..."
                    )
                    time.sleep(wait_seconds)
                    continue
                return {
                    "answer": (
                        "⚠️ CONNECTION TIMEOUT: The network/API stream timed out after "
                        f"{max_attempts} attempts. Please retry in a few seconds."
                    ),
                    "sources": [],
                    "retrieval_mode": "timeout",
                }
            if "422" in error_msg:
                return {
                    "answer": "⚠️ MODEL ERROR: Granite 3.1 is currently busy. Wait 10 seconds and retry.",
                    "sources": [],
                    "retrieval_mode": "model-error",
                }
            return {
                "answer": f"⚠️ SYSTEM ERROR: {e}",
                "sources": [],
                "retrieval_mode": "system-error",
            }

print("\n" + "=" * 52)
print("🎓 ACADEMIC RESEARCH ENGINE: ONLINE")
print("Target: Full Essay Output (3 Intro + 6 Body + 3 Conclusion)")
print(f"Mode: Hierarchical Hybrid Retrieval + PDF-Grounded Essay ({RUNTIME_MODE.upper()})")
print(f"Visual Panel: {'ON' if VISUAL_PANEL_ON else 'OFF'} | Auto Brief: {'ON' if AUTO_BRIEF_ON else 'OFF'} | Auto Export: {'ON' if AUTO_BRIEF_EXPORT_ON else 'OFF'}")
print(f"Brief Style Key: {BRIEF_STYLE_KEY}")
print(f"Network LLM Synthesis: {'ON — Essay Mode Active' if ENABLE_LLM_SYNTHESIS and ENABLE_NETWORK_LLM else 'OFF (local evidence only)'}")
print(f"Fusion Pack: {'ON' if USE_FUSION_PACK else 'OFF'}")
print("=" * 52)
print("Type 'end' to exit.")

while True:
    user_input = input("\n🟦 Research Question: ").strip()

    if not user_input:
        continue
    if user_input.lower() in ["end", "exit", "quit", "stop"]:
        print("🛑 Closing Research Engine. Good luck with your assignment!")
        break

    if RUNTIME_MODE == "fast":
        print("🔍 Running retrieval and generating full essay... (this may take 30-90 seconds)")
    else:
        print("🔍 Running deep retrieval and generating full essay... (this may take 60-120 seconds)")

    start_time = time.time()
    result = run_academic_query(user_input, max_attempts=3)
    duration = time.time() - start_time

    answer = result.get("answer", "")
    sources = result.get("sources", [])
    retrieval_mode = result.get("retrieval_mode", "unknown")

    # Auto-render styled notebook panel when visual helpers are available.
    if VISUAL_PANEL_ON and "render_answer_panel" in globals():
        try:
            render_answer_panel(user_input, result)
        except Exception as render_error:
            print(f"⚠️ Visual panel render failed: {render_error}")

    # Optional: auto-generate a style-driven slide brief per answer.
    if AUTO_BRIEF_ON and "build_slide_brief" in globals():
        try:
            chosen_style = BRIEF_STYLE_KEY
            if "style_library" in globals():
                curated_keys = list((style_library or {}).get("curated", {}).keys())
                if curated_keys and chosen_style not in curated_keys:
                    chosen_style = curated_keys[0]

            brief = build_slide_brief(user_input, result, style_key=chosen_style)
            print(f"🧾 Slide brief generated (style={chosen_style}).")

            if AUTO_BRIEF_EXPORT_ON and "export_slide_brief" in globals():
                try:
                    brief_path = export_slide_brief(brief)
                    print(f"💾 Slide brief saved: {brief_path}")
                except Exception as export_error:
                    print(f"⚠️ Slide brief export failed: {export_error}")
        except Exception as brief_error:
            print(f"⚠️ Slide brief generation failed: {brief_error}")

    print("\n" + "=" * 60)
    print("📝 ACADEMIC ESSAY (PDF-Grounded | Granite 3.1 + MemPalace)")
    print("=" * 60)
    print(answer)
    print("=" * 60)
    print(f"⏱️ Essay completed in {duration:.1f} seconds.")
    print(f"🧭 Retrieval Mode: {retrieval_mode}")

    print("\n📍 SOURCES:")
    if not sources:
        print("- No traceable source nodes returned for this answer.")
    else:
        for ref in sources[:SOURCE_PRINT_LIMIT]:
            snippet = ref["snippet"] or "No snippet available"
            print(
                f"- [{ref['retriever']}] {ref['file_name']} | "
                f"wing={ref['wing']} | room={ref['room']} | hall={ref['hall']} | drawer={ref['drawer_type']} | "
                f"\"{snippet}...\""
            )



🎓 ACADEMIC RESEARCH ENGINE: ONLINE
Target: Evidence-first, traceable cross-document grounding
Mode: Hierarchical Hybrid Retrieval + Citation Output (FAST)
Visual Panel: ON | Auto Brief: ON | Auto Export: OFF
Brief Style Key: artifact
Network LLM Synthesis: OFF (local fast mode)
Fusion Pack: OFF
Type 'end' to exit.


🔍 Running fast hierarchical retrieval... (typically 3-12 seconds)


🧾 Slide brief generated (style=artifact).

🧠 FACTUAL ANALYSIS (Granite 3.1 + MemPalace):
------------------------------
1) SUMMARY
- Fast mode active: returning evidence-first extract without full synthesis.
- Question: What is adaptive leadership

2) EVIDENCE-TRACE TABLE
- Claim: Candidate evidence from fusion retriever
  Quote: "Fusion pack disabled or no direct evidence returned."
  Source: Current Document|Palace | file=Unknown | wing=Projects | room=General_Topic | hall=Research_Evidence | drawer=PDF_Text
- Claim: Candidate evidence from current document
  Quote: "The  adaptive problem of changing employee mindsets from job stability to ongoing  learning and creativity must be addressed by adaptive KM, which goes beyond  simply integrating knowledge-sharing systems.

The role of adaptive leadership, cultural competence, resilience, and  knowledge management in effectively managing InnovTech’s  organizational transformation.    InnovoTech Solutions is a rapidly expanding multinatio